# 04 â€” Training: YOLOv8s (one-stage)

Model 2/3. Protocol shared with 03/05: same split (`fathomnet-02-split`, seed 42), same unified COCO evaluation (pycocotools, score thr 0.01, max 300 dets), same head/mid/tail cuts. Native input policy: imgsz 960. COCO-pretrained, 30 epochs.

Outputs: `results_yolov8s.json`, `preds_yolov8s.json`, ultralytics `results.csv` + curve plots, weights.

In [ ]:
%pip install -q ultralytics

In [ ]:
# --- 1. Setup, data, shared eval helpers (kept identical across 03/04/05) ---
import os, json, glob, time, random, io, contextlib
from pathlib import Path
import numpy as np, pandas as pd
import torch
import matplotlib as mpl, matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', font_scale=1.1)
from ultralytics import YOLO
from pycocotools.coco import COCO

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT = Path('/kaggle/working')

anns_path = Path(glob.glob('/kaggle/input/**/train_dataset.json', recursive=True)[0])
ANN = anns_path.parent
IMG = ANN.parent / 'images' if (ANN.parent / 'images').exists() else ANN / 'images'
data = json.load(open(anns_path))
cats = {c['id']: c['name'] for c in data['categories']}
CAT_IDS = sorted(cats)
img_by_id = {i['id']: i for i in data['images']}
anns_by_img = {}
for a in data['annotations']:
    anns_by_img.setdefault(a['image_id'], []).append(a)

# val split: prefer kernel_sources-mounted split.json; if not mounted, make a local 90/10
try:
    sp = glob.glob('/kaggle/input/**/split.json', recursive=True)
    if sp:
        S = json.load(open(Path(sp[0])))
        val_ids = S['val_ids']
        train_ids = set(S['train_ids'])
    else:
        raise FileNotFoundError('split.json glob returned nothing')
except Exception as e:
    print('WARNING: split.json not available (%s). Creating a local 90/10 split (seed=%d).' % (e, SEED))
    all_ids = sorted(img_by_id.keys())
    rng = random.Random(SEED)
    rng.shuffle(all_ids)
    n_val = max(1, int(len(all_ids) * 0.1))
    val_ids = all_ids[:n_val]
    train_ids = set(all_ids[n_val:])
    S = {'val_ids': val_ids, 'train_ids': sorted(train_ids), 'val_fraction': 0.1}
val_list = sorted(val_ids)

from pycocotools.cocoeval import COCOeval
GT = {'images': [img_by_id[i] for i in val_ids],
      'annotations': [a for a in data['annotations'] if a['image_id'] in set(val_ids)],
      'categories': data['categories']}
json.dump(GT, open(OUT / 'val_gt.json', 'w'))
_coco_sil = io.StringIO()
with contextlib.redirect_stdout(_coco_sil):
    GT_COCO = COCO(str(OUT / 'val_gt.json'))


def coco_eval(preds, tag):
    'Unified COCO bbox eval used by 03/04/05. Returns mAP, mAP50, per-class AP.'
    dt = GT_COCO.loadRes(preds)
    E = COCOeval(GT_COCO, dt, 'bbox')
    with contextlib.redirect_stdout(io.StringIO()):
        E.evaluate(); E.accumulate(); E.summarize()
    mAP, mAP50 = float(E.stats[0]), float(E.stats[1])
    per = {}
    for cid in CAT_IDS:
        E.params.catIds = [cid]
        with contextlib.redirect_stdout(io.StringIO()):
            E.evaluate(); E.accumulate(); E.summarize()
        per[cats[cid]] = round(float(E.stats[0]), 4)
    E.params.catIds = []
    return {'mAP50-95': round(mAP, 4), 'mAP50': round(mAP50, 4), 'per_class_AP50-95': per}


_counts = pd.Series([a['category_id'] for a in data['annotations']]).map(cats).value_counts()
GROUPS = {'head': set(_counts.index[:3]), 'mid': set(_counts.index[3:7]), 'tail': set(_counts.index[7:])}


def group_ap(per_class):
    return {g: round(float(np.mean([per_class[c] for c in ids if c in per_class])), 4)
            for g, ids in GROUPS.items()}



print('data loaded; val=%d train=%d images; cats=%d' % (len(val_list), len(train_ids), len(CAT_IDS)))


In [ ]:
# --- 2. Build YOLO-format dataset (symlinked images + normalized txt labels) ---
BASE = Path('/kaggle/working/dataset')
# ultralytics needs contiguous class indices: map original cat ids -> 0..31, remember reverse
IDX = {cid: k for k, cid in enumerate(CAT_IDS)}

def img_size(info):
    if info.get('width'):
        return info['width'], info['height']
    with Image.open(IMG / info['file_name']) as im:
        return im.size

for sp, ids in [('train', sorted(train_ids)), ('val', val_list)]:
    (BASE / 'images' / sp).mkdir(parents=True, exist_ok=True)
    (BASE / 'labels' / sp).mkdir(parents=True, exist_ok=True)
    for iid in ids:
        info = img_by_id[iid]
        src = IMG / info['file_name']
        dst = BASE / 'images' / sp / Path(info['file_name'])
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            os.symlink(src, dst)
        w, h = img_size(info)
        lines = []
        for a in anns_by_img.get(iid, []):
            x, y, bw, bh = a['bbox']
            cx, cy = (x + bw / 2) / w, (y + bh / 2) / h
            nw, nh = min(bw / w, 1.0), min(bh / h, 1.0)
            if nw <= 0 or nh <= 0:
                continue
            lines.append(f'{IDX[a["category_id"]]} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
        lp = Path(str(dst).replace('/images/', '/labels/')).with_suffix('.txt')
        lp.write_text('\n'.join(lines))

names = {IDX[cid]: cats[cid] for cid in CAT_IDS}
yaml_p = BASE / 'fathomnet.yaml'
yaml_p.write_text(
    f'path: {BASE}\ntrain: images/train\nval: images/val\nnc: {len(CAT_IDS)}\n'
    'names:\n' + '\n'.join(f'  {k}: {v}' for k, v in names.items()))
print('dataset built:', yaml_p)

In [ ]:
# --- 3. Train YOLOv8s ---
model = YOLO('yolov8s.pt')
model.train(data=str(yaml_p), epochs=30, imgsz=960, batch=8, seed=SEED,
            deterministic=True, workers=2, project=str(OUT / 'runs'), name='yolov8s',
            patience=50, exist_ok=True)

In [ ]:
# --- 4. Predict on val + unified COCO evaluation ---
best = YOLO(str(OUT / 'runs' / 'yolov8s' / 'weights' / 'best.pt'))
N_PARAMS = sum(p.numel() for p in best.model.parameters())

path2id = {str(BASE / 'images' / 'val' / Path(img_by_id[i]['file_name'])): i for i in val_list}
preds = []
# NOTE: pass the val FOLDER streamed (stream=True), never a Python list of all paths.
# With a list source the dataloader stacks all images into one warmup batch,
# so warmup allocates the whole 646-image tensor (~17.7 GiB at imgsz 960) and dies on a T4.
# (batch=8 below is only the internal inference micro-batch; training is NOT re-run.
#  Kept at the same output size as before.)
for r in best.predict(source=str(BASE / 'images' / 'val'), conf=0.001, iou=0.7,
                      max_det=300, imgsz=960, batch=8, stream=True, verbose=False):
    iid = path2id[str(r.path)]
    xyxy = r.boxes.xyxy.cpu().numpy(); sc = r.boxes.conf.cpu().numpy(); cl = r.boxes.cls.cpu().numpy()
    for (x1, y1, x2, y2), s, c in zip(xyxy, sc, cl):
        preds.append({'image_id': iid, 'category_id': CAT_IDS[int(c)],
                      'bbox': [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                      'score': float(s)})
    torch.cuda.empty_cache()
json.dump(preds, open(OUT / 'preds_yolov8s.json', 'w'))
print('detections:', len(preds))

R = {'model': 'yolov8s', 'protocol': {'seed': SEED, 'val_split': S['val_fraction'], 'epochs': 30,
     'input': 'imgsz=960', 'score_thr': 0.001, 'max_det': 300},
     'params_million': round(N_PARAMS / 1e6, 2), **coco_eval(preds, 'yolov8s')}
R['head_mid_tail_AP50-95'] = group_ap(R['per_class_AP50-95'])

# latency (median of 30 single-image inferences at native imgsz)
import cv2
x0 = cv2.imread(str(BASE / 'images' / 'val' / Path(img_by_id[val_list[0]]['file_name'])))
for _ in range(5):
    best.predict(x0, imgsz=960, verbose=False)
ts = []
for _ in range(30):
    t0 = time.time(); best.predict(x0, imgsz=960, verbose=False); ts.append(time.time() - t0)
R['latency_ms_per_image'] = round(float(np.median(ts)) * 1000, 2)

json.dump(R, open(OUT / 'results_yolov8s.json', 'w'), indent=2)
print(json.dumps({k: R[k] for k in ['mAP50-95', 'mAP50', 'head_mid_tail_AP50-95',
      'latency_ms_per_image']}, indent=2))

In [ ]:
# --- 5. Artifacts: ultralytics log, curves ---
src_csv = OUT / 'runs' / 'yolov8s' / 'results.csv'
if src_csv.exists():
    h = pd.read_csv(src_csv); h.columns = [c.strip() for c in h.columns]
    h.to_csv(OUT / 'curve_yolov8s.csv', index=False)
    mcols = [c for c in h.columns if 'mAP50-95' in c]
    lcols = [c for c in h.columns if c.endswith('loss') and 'box_loss' in c]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    if lcols:
        axes[0].plot(h.epoch, h[lcols[0]], marker='o', color='#4c72b0')
        axes[0].set(title='YOLOv8s box_loss', xlabel='epoch')
    if mcols:
        axes[1].plot(h.epoch, h[mcols[0]], marker='o', color='#55a868')
        axes[1].set(title='YOLOv8s mAP50-95 (val)', xlabel='epoch')
    fig.tight_layout()
    for ext in ['png', 'pdf']:
        fig.savefig(OUT / f'curve_yolov8s.{ext}', dpi=200)
    plt.show()
print('done')